# Emotion Recognition with Enhanced Att-1DCNN-GRU + TACO Cross-Attention

**Paper:** "Emotion recognition based on multimodal physiological electrical signals"

**Dataset:** DREAMER (23 subjects, 18 trials, 14-ch EEG @ 128Hz, 2-ch ECG @ 256Hz)

**Pipeline:**
1. Clone repo from GitHub & install dependencies
2. Data Loading & Preprocessing (filtering, ICA, segmentation)
3. 1D→2D Transforms (GAF/RP/MTF for ECG, spatial grid for EEG)
4. Model Build (Transformer Encoders + TACO Cross-Attention Fusion)
5. LOSOCV Training & Evaluation

## 0. Clone Repo & Setup

In [ ]:
# ── Install Git LFS & Clone full repository ──────────────────────────────────
# Replace with your GitHub repo URL
REPO_URL = 'https://github.com/Sisoodiya/Sanjay-sWork.git'  # <-- CHANGE THIS
REPO_DIR = '/content/emotion-recognition'

# Install Git LFS
!sudo apt-get update -qq
!sudo apt-get install -y -qq git git-lfs
!git lfs install

import os

# Remove existing folder if present
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

# Clone the full repository
!git clone {REPO_URL} {REPO_DIR}

# Enter repo and pull all LFS files
os.chdir(REPO_DIR)
!git lfs pull

print(f'Working directory: {os.getcwd()}')
!ls -la
!ls -la data/

In [ ]:
# ── Install dependencies + CuPy for GPU-accelerated preprocessing ────────────
!pip install -q -r requirements.txt
!pip install -q cupy-cuda12x

In [ ]:
# ── Verify dataset ───────────────────────────────────────────────────────────────
# DREAMER.mat should have been pulled automatically via Git LFS
assert os.path.exists('data/DREAMER.mat'), 'DREAMER.mat not found! Run: git lfs pull'
file_size = os.path.getsize('data/DREAMER.mat') / (1024 * 1024)
print(f'DREAMER.mat ready ({file_size:.0f} MB)')

# If Git LFS only downloaded a pointer file (< 1 MB), pull the actual data
if file_size < 1:
    print('LFS pointer detected, pulling actual data...')
    !git lfs pull
    file_size = os.path.getsize('data/DREAMER.mat') / (1024 * 1024)
    print(f'DREAMER.mat ready ({file_size:.0f} MB)')

## 1. Imports & Verification

In [ ]:
import sys
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure repo root is on the Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Import from src/ modules
from src import config
from src.utils import set_seed, get_class_weights, ensure_dir, HAS_CUPY
from src.data_loader import load_all_subjects, get_labels
from src.preprocessing import (
    preprocess_eeg, preprocess_ecg,
    extract_last_n_seconds, segment_signal, build_dataset,
)
from src.transforms import (
    ecg_to_2d, eeg_to_2d_grid,
    transform_ecg_batch, transform_eeg_batch,
)
from src.model import build_model
from src.evaluate import (
    compute_metrics, plot_confusion_matrix,
    print_classification_report, print_results_table,
)
from train import prepare_fold_data, apply_transforms, losocv_train

set_seed(config.RANDOM_SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print(f'CuPy acceleration: {"enabled" if HAS_CUPY else "disabled (falling back to NumPy)"}')
print(f'Data path: {config.DATA_PATH}')
print(f'Save dir:  {config.SAVE_DIR}')

## 2. Load & Preprocess Data

In [ ]:
# Build the full preprocessed dataset (all 23 subjects)
# This applies: notch filter -> bandpass -> ICA -> last 60s -> 1s segments
print('Building preprocessed dataset...')
dataset = build_dataset()

total = sum(d['eeg_segments'].shape[0] for d in dataset)
print(f'\nTotal segments: {total} '
      f'(expected ~{config.NUM_SUBJECTS * config.NUM_TRIALS * config.LAST_SECONDS})')

## 3. Visualize 2D Transforms

In [ ]:
# ── ECG 2D Transforms (GAF, RP, MTF) ─────────────────────────────────────────
sample_ecg = dataset[0]['ecg_segments'][0]  # (256, 2)
sample_2d = ecg_to_2d(sample_ecg)

titles = ['Ch1 GAF', 'Ch1 RP', 'Ch1 MTF', 'Ch2 GAF', 'Ch2 RP', 'Ch2 MTF']
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for idx, ax in enumerate(axes.flat):
    ax.imshow(sample_2d[:, :, idx], cmap='viridis', aspect='auto')
    ax.set_title(titles[idx])
    ax.axis('off')
plt.suptitle('ECG 2D Transforms (Subject 0, Segment 0)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── EEG 9x9 Spatial Grid ────────────────────────────────────────────────────────
sample_eeg = dataset[0]['eeg_segments'][0]  # (128, 14)
sample_grid = eeg_to_2d_grid(sample_eeg)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(sample_grid[0], cmap='RdBu_r', aspect='auto')
ax.set_title('EEG 9x9 Spatial Grid (t=0)')
for ch_idx, (row, col) in config.EEG_GRID_MAP.items():
    ax.text(col, row, config.EEG_ELECTRODE_NAMES[ch_idx], ha='center', va='center',
            fontsize=7, color='black', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Build & Verify Model

In [ ]:
model = build_model()
model.summary()

# Quick forward pass test
test_eeg = np.random.randn(2, 128, 9, 9).astype(np.float32)
test_ecg = np.random.randn(2, 64, 64, 6).astype(np.float32)
out = model.predict([test_eeg, test_ecg], verbose=0)
print(f'\nForward pass: EEG {test_eeg.shape} + ECG {test_ecg.shape} -> {out.shape}')
print(f'Predictions: {out}')

# Clean up test model
del model
tf.keras.backend.clear_session()

## 5. LOSOCV Training

Leave-One-Subject-Out Cross-Validation: train on 22 subjects, test on 1, repeat for all 23.

### Quick Test (2 Subjects)

In [ ]:
# Quick test with 2 subjects to verify everything works
quick_results = {}
for target in config.TARGETS:
    print(f"\n{'#' * 70}")
    print(f'  TRAINING: {target.upper()}')
    print(f"{'#' * 70}")
    quick_results[target] = losocv_train(dataset, target=target, num_subjects=2)

print_results_table(quick_results)

### Full LOSOCV (All 23 Subjects)

Uncomment and run for the complete experiment. This takes several hours on GPU.

In [ ]:
# all_results_full = {}
# for target in config.TARGETS:
#     print(f"\n{'#' * 70}")
#     print(f'  TRAINING: {target.upper()}')
#     print(f"{'#' * 70}")
#     all_results_full[target] = losocv_train(dataset, target=target, num_subjects=None)
#
# print_results_table(all_results_full)